# 03 — Tool Use

This notebook demonstrates Claude's tool use capabilities in a coding context —
defining tools, handling tool calls, and building a multi-step tool chain.

## Learning Objectives
1. Define tool schemas for the Anthropic API
2. Handle tool_use responses and return tool_results
3. Build a multi-step tool chain that reads, modifies, and tests code

## Setup

In [ ]:
import os
import json
import subprocess
from pathlib import Path
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic()
print("Client initialized.")

## Defining Coding Tools

Each tool needs a name, description, and JSON Schema for its input parameters.

In [ ]:
CODING_TOOLS = [
    {
        "name": "read_file",
        "description": "Read the contents of a file from disk.",
        "input_schema": {
            "type": "object",
            "properties": {
                "file_path": {"type": "string", "description": "Path to the file to read."}
            },
            "required": ["file_path"],
        },
    },
    {
        "name": "write_file",
        "description": "Write content to a file. Creates the file if it doesn't exist.",
        "input_schema": {
            "type": "object",
            "properties": {
                "file_path": {"type": "string", "description": "Path to the file."},
                "content": {"type": "string", "description": "Content to write."},
            },
            "required": ["file_path", "content"],
        },
    },
    {
        "name": "run_command",
        "description": "Execute a shell command and return stdout/stderr.",
        "input_schema": {
            "type": "object",
            "properties": {
                "command": {"type": "string", "description": "The shell command to run."}
            },
            "required": ["command"],
        },
    },
    {
        "name": "search_code",
        "description": "Search for a pattern in files within a directory.",
        "input_schema": {
            "type": "object",
            "properties": {
                "pattern": {"type": "string", "description": "Search pattern (regex)."},
                "directory": {"type": "string", "description": "Directory to search in."},
            },
            "required": ["pattern", "directory"],
        },
    },
]

print(f"Defined {len(CODING_TOOLS)} tools.")

## Tool Executor

In [ ]:
WORKSPACE = Path("/tmp/tool_use_demo")
WORKSPACE.mkdir(exist_ok=True)


def execute_tool(name: str, params: dict) -> str:
    """Execute a tool and return the result."""
    match name:
        case "read_file":
            p = WORKSPACE / params["file_path"]
            return p.read_text() if p.exists() else f"Error: {p} not found"

        case "write_file":
            p = WORKSPACE / params["file_path"]
            p.parent.mkdir(parents=True, exist_ok=True)
            p.write_text(params["content"])
            return f"Written to {p}"

        case "run_command":
            r = subprocess.run(
                params["command"], shell=True,
                capture_output=True, text=True,
                cwd=str(WORKSPACE), timeout=30,
            )
            return r.stdout + (f"\nSTDERR: {r.stderr}" if r.returncode != 0 else "")

        case "search_code":
            d = WORKSPACE / params["directory"]
            r = subprocess.run(
                ["grep", "-rn", params["pattern"], str(d)],
                capture_output=True, text=True,
            )
            return r.stdout or "No matches found."

        case _:
            return f"Unknown tool: {name}"


print("Tool executor ready.")

## Create Sample Files

In [ ]:
# Create a sample Python file for Claude to work with
(WORKSPACE / "calculator.py").write_text("""\
def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

def divide(a, b):
    return a / b
""")

print("Sample calculator.py created in workspace.")

## The Tool Use Loop

In [ ]:
def tool_use_loop(user_request: str, max_turns: int = 10) -> str:
    """Run a complete Claude tool use conversation."""
    messages = [{"role": "user", "content": user_request}]

    for turn in range(max_turns):
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=4096,
            system=(
                "You are a coding assistant with access to file and command tools. "
                f"The workspace directory is {WORKSPACE}. "
                "Use relative paths when calling tools."
            ),
            tools=CODING_TOOLS,
            messages=messages,
        )

        print(f"\nTurn {turn + 1} | stop_reason={response.stop_reason}")

        if response.stop_reason == "end_turn":
            return "\n".join(b.text for b in response.content if b.type == "text")

        # Handle tool calls
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []

        for block in response.content:
            if block.type == "tool_use":
                print(f"  → {block.name}({json.dumps(block.input, indent=2)})")
                result = execute_tool(block.name, block.input)
                print(f"  ← {result[:150]}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result,
                })

        messages.append({"role": "user", "content": tool_results})

    return "Max turns reached."


print("Tool use loop ready.")

In [ ]:
# Ask Claude to review and improve the calculator
result = tool_use_loop(
    "Read calculator.py, add type hints to all functions, "
    "add error handling to the divide function (raise ValueError for zero), "
    "and write the updated file."
)

print("\n=== FINAL RESPONSE ===")
print(result)

In [ ]:
# Verify the updated file
print((WORKSPACE / "calculator.py").read_text())

## Exercise

1. Ask Claude to create a test file for `calculator.py` using the tools
2. Ask Claude to run the tests with `python -m pytest`
3. Track which tools Claude uses and in what order — note the tool chain pattern